# Midpoint vs Trapezoid Along the Characteristic

This notebook is a numerical design study, not a feature tour of active `jact` solver modes. The production solver now uses midpoint quadrature only. We keep trapezoid here as a local reference method for the 1-D hazard integral along the transported characteristic.


In [ ]:
import math
import numpy as np

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt

import jact


## Setup

We use a two-state model with one outgoing transition `healthy -> dead`. Because there is only one hazard, the exact state probabilities reduce to the survival curve along the characteristic `d(t) = d_0 + t`.


In [ ]:
HORIZON = 2
INITIAL_DURATION = 0.3
REFINEMENTS = [4, 8, 16, 32, 64, 128]
REPRESENTATIVE_STEPS = 16

state_space = jact.StateSpace(
    states=["healthy", "dead"],
    transitions=[("healthy", "dead")],
)


def smooth_intensity(t, d, **kwargs):
    return 0.12 + 0.04 * t + 0.03 * d + 0.015 * (t + d) ** 2


def smooth_cumulative_hazard(times, d_0):
    c0 = 0.12 + 0.03 * d_0 + 0.015 * d_0**2
    c1 = 0.07 + 0.06 * d_0
    c2 = 0.06
    return c0 * times + 0.5 * c1 * times**2 + (c2 / 3.0) * times**3


def jump_intensity(t, d, **kwargs):
    base = 0.10 + 0.05 * d + 0.01 * (t + d) ** 2
    jump = jnp.where(t < 1.0, 0.0, 0.18)
    return base + jump


def jump_cumulative_hazard(times, d_0):
    c0 = 0.10 + 0.05 * d_0 + 0.01 * d_0**2
    c1 = 0.05 + 0.04 * d_0
    c2 = 0.04
    smooth_part = c0 * times + 0.5 * c1 * times**2 + (c2 / 3.0) * times**3
    jump_part = 0.18 * jnp.maximum(times - 1.0, 0.0)
    return smooth_part + jump_part


def probability_from_cumulative_hazard(cumulative_hazard_fn, times, d_0):
    survival = jnp.exp(-cumulative_hazard_fn(times, d_0))
    return jnp.stack([survival, 1.0 - survival], axis=-1)


smooth_model = state_space.build(
    transitions={("healthy", "dead"): smooth_intensity}
)
jump_model = state_space.build(
    transitions={("healthy", "dead"): jump_intensity}
)


In [ ]:
def solve_probabilities(model, steps_per_unit, horizon=HORIZON, initial_duration=INITIAL_DURATION):
    result = model.solve(
        initial="healthy",
        initial_duration=initial_duration,
        horizon=horizon,
        steps_per_unit=steps_per_unit,
        probability="collapse_point_no_duration",
    )
    assert result["states"] == ("healthy", "dead")
    times = jnp.linspace(0.0, horizon, horizon * steps_per_unit + 1)
    return times, result["probability"][:, 0, :]


def trapezoid_probabilities(intensity_fn, steps_per_unit, horizon=HORIZON, initial_duration=INITIAL_DURATION):
    dt = 1.0 / steps_per_unit
    times = jnp.linspace(0.0, horizon, horizon * steps_per_unit + 1)
    left_t = times[:-1]
    right_t = times[1:]
    left_d = (initial_duration + left_t)[None, :]
    right_d = (initial_duration + right_t)[None, :]
    left = jnp.squeeze(intensity_fn(left_t, left_d), axis=0)
    right = jnp.squeeze(intensity_fn(right_t, right_d), axis=0)
    hazards = 0.5 * dt * (left + right)
    cumulative = jnp.concatenate([jnp.zeros((1,)), jnp.cumsum(hazards)])
    survival = jnp.exp(-cumulative)
    probability = jnp.stack([survival, 1.0 - survival], axis=-1)
    return times, probability


def max_probability_error(numeric, exact):
    return float(jnp.max(jnp.abs(numeric - exact)))


def observed_orders(errors):
    orders = [np.nan]
    for previous, current in zip(errors[:-1], errors[1:]):
        orders.append(math.log(previous / current, 2.0))
    return orders


def run_study(solve_fn, analytic_probability_fn):
    rows = []
    runs = {}
    for steps_per_unit in REFINEMENTS:
        times, numeric = solve_fn(steps_per_unit)
        exact = analytic_probability_fn(times, INITIAL_DURATION)
        rows.append(
            {
                "steps_per_unit": steps_per_unit,
                "step_size": 1.0 / steps_per_unit,
                "max_error": max_probability_error(numeric, exact),
            }
        )
        runs[steps_per_unit] = {
            "times": times,
            "numeric": numeric,
            "exact": exact,
        }

    errors = [row["max_error"] for row in rows]
    for row, order in zip(rows, observed_orders(errors)):
        row["observed_order"] = order

    return {"rows": rows, "runs": runs}


def print_study(title, study):
    print(title)
    print(f"{'steps/unit':>10} {'dt':>10} {'max error':>14} {'observed p':>12}")
    for row in study["rows"]:
        order = "--" if np.isnan(row["observed_order"]) else f"{row['observed_order']:.4f}"
        print(
            f"{row['steps_per_unit']:>10d} "
            f"{row['step_size']:>10.5f} "
            f"{row['max_error']:>14.6e} "
            f"{order:>12}"
        )


def reference_second_order(step_sizes, errors):
    return errors[0] * (step_sizes / step_sizes[0]) ** 2


def reference_first_order(step_sizes, errors):
    return errors[0] * (step_sizes / step_sizes[0])


## Smooth case

For a smooth hazard, both midpoint and trapezoid are second-order along the characteristic. The midpoint curve comes from the actual `jact` solver. The trapezoid curve is computed locally in this notebook.


In [ ]:
smooth_midpoint_study = run_study(
    lambda steps: solve_probabilities(smooth_model, steps),
    lambda times, d_0: probability_from_cumulative_hazard(smooth_cumulative_hazard, times, d_0),
)
smooth_trapezoid_study = run_study(
    lambda steps: trapezoid_probabilities(smooth_intensity, steps),
    lambda times, d_0: probability_from_cumulative_hazard(smooth_cumulative_hazard, times, d_0),
)

print_study("Smooth hazard: midpoint solver", smooth_midpoint_study)
print()
print_study("Smooth hazard: local trapezoid reference", smooth_trapezoid_study)


In [ ]:
smooth_mid_run = smooth_midpoint_study["runs"][REPRESENTATIVE_STEPS]
smooth_trap_run = smooth_trapezoid_study["runs"][REPRESENTATIVE_STEPS]
smooth_step_sizes = np.array([row["step_size"] for row in smooth_midpoint_study["rows"]])
smooth_mid_errors = np.array([row["max_error"] for row in smooth_midpoint_study["rows"]])
smooth_trap_errors = np.array([row["max_error"] for row in smooth_trapezoid_study["rows"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
times = np.asarray(smooth_mid_run["times"])
exact = np.asarray(smooth_mid_run["exact"])
midpoint = np.asarray(smooth_mid_run["numeric"])
trapezoid = np.asarray(smooth_trap_run["numeric"])

axes[0].plot(times, exact[:, 0], color="black", linewidth=2.0, label="analytic healthy")
axes[0].plot(times, midpoint[:, 0], "--", linewidth=2.0, label="midpoint solver")
axes[0].plot(times, trapezoid[:, 0], ":", linewidth=2.5, label="local trapezoid")
axes[0].set_title(f"Smooth hazard at {REPRESENTATIVE_STEPS} steps/unit")
axes[0].set_xlabel("time")
axes[0].set_ylabel("healthy probability")
axes[0].legend()

axes[1].loglog(smooth_step_sizes, smooth_mid_errors, "o-", linewidth=2.0, label="midpoint solver")
axes[1].loglog(smooth_step_sizes, smooth_trap_errors, "s-", linewidth=2.0, label="local trapezoid")
axes[1].loglog(
    smooth_step_sizes,
    reference_second_order(smooth_step_sizes, smooth_mid_errors),
    "k--",
    linewidth=1.5,
    label="$O(h^2)$ reference",
)
axes[1].set_title("Smooth-case max probability error")
axes[1].set_xlabel("step size h = 1 / steps_per_unit")
axes[1].set_ylabel("max absolute error")
axes[1].legend()

fig.tight_layout()


## Grid-aligned jump case

Now introduce a jump in time at `t = 1.0`. The midpoint path is still the production solver. Trapezoid is again only a local reference computation. We keep powers of two for `steps_per_unit`, so the jump always lies exactly on the grid.


In [ ]:
jump_midpoint_study = run_study(
    lambda steps: solve_probabilities(jump_model, steps),
    lambda times, d_0: probability_from_cumulative_hazard(jump_cumulative_hazard, times, d_0),
)
jump_trapezoid_study = run_study(
    lambda steps: trapezoid_probabilities(jump_intensity, steps),
    lambda times, d_0: probability_from_cumulative_hazard(jump_cumulative_hazard, times, d_0),
)

print_study("Grid-aligned jump: midpoint solver", jump_midpoint_study)
print()
print_study("Grid-aligned jump: local trapezoid reference", jump_trapezoid_study)


In [ ]:
jump_mid_run = jump_midpoint_study["runs"][REPRESENTATIVE_STEPS]
jump_trap_run = jump_trapezoid_study["runs"][REPRESENTATIVE_STEPS]
jump_step_sizes = np.array([row["step_size"] for row in jump_midpoint_study["rows"]])
jump_mid_errors = np.array([row["max_error"] for row in jump_midpoint_study["rows"]])
jump_trap_errors = np.array([row["max_error"] for row in jump_trapezoid_study["rows"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
times = np.asarray(jump_mid_run["times"])
exact = np.asarray(jump_mid_run["exact"])
midpoint = np.asarray(jump_mid_run["numeric"])
trapezoid = np.asarray(jump_trap_run["numeric"])

axes[0].plot(times, exact[:, 0], color="black", linewidth=2.0, label="analytic healthy")
axes[0].plot(times, midpoint[:, 0], "--", linewidth=2.0, label="midpoint solver")
axes[0].plot(times, trapezoid[:, 0], ":", linewidth=2.5, label="local trapezoid")
axes[0].axvline(1.0, color="gray", linestyle=":", linewidth=1.5, label="jump at t = 1")
axes[0].set_title(f"Grid-aligned jump at {REPRESENTATIVE_STEPS} steps/unit")
axes[0].set_xlabel("time")
axes[0].set_ylabel("healthy probability")
axes[0].legend()

axes[1].loglog(jump_step_sizes, jump_mid_errors, "o-", linewidth=2.0, label="midpoint solver")
axes[1].loglog(jump_step_sizes, jump_trap_errors, "s-", linewidth=2.0, label="local trapezoid")
axes[1].loglog(
    jump_step_sizes,
    reference_second_order(jump_step_sizes, jump_mid_errors),
    "k--",
    linewidth=1.5,
    label="$O(h^2)$ reference",
)
axes[1].loglog(
    jump_step_sizes,
    reference_first_order(jump_step_sizes, jump_trap_errors),
    "k:",
    linewidth=1.5,
    label="$O(h)$ reference",
)
axes[1].set_title("Grid-aligned jump: max probability error")
axes[1].set_xlabel("step size h = 1 / steps_per_unit")
axes[1].set_ylabel("max absolute error")
axes[1].legend()

fig.tight_layout()


## Takeaways

- Midpoint is second-order on the smooth example and competitive with trapezoid in error constant.
- On the grid-aligned jump example, midpoint remains second-order while the local trapezoid reference is first-order.
- This is why `jact` keeps midpoint as the production rule and documents grid alignment as the main practical mitigation for piecewise hazards.
